# Chapitre 17 - Modèles graphiques non orientés
## Cours 2 : variables binaires - modèles de Ising et machines de Boltzmann

Ce notebook couvre la section 17.4 du livre : les graphes de Markov par paires dont
les sommets sont des variables **binaires** $X_j \in \{0, 1\}$. Ces modèles s'appellent
**modèles de Ising** en physique statistique et **machines de Boltzmann** en
apprentissage automatique (les sommets y sont des « unités »).

Plan :

1. le modèle de Ising, la fonction de partition et le nœud constant $X_0 \equiv 1$ ;
2. les lois conditionnelles logistiques et l'interprétation des paramètres ;
3. le maximum de vraisemblance : égalisation des moments, Newton et IPF par
   **énumération** des $2^p$ états (petit $p$) ;
4. quand $p$ est grand : échantillonnage de Gibbs, champ moyen, **pseudo-vraisemblance**
   et régressions logistiques $\ell_1$ nœud par nœud (estimation du graphe) ;
5. les nœuds cachés et les **machines de Boltzmann restreintes** (RBM), avec un exemple
   sur les chiffres manuscrits.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import ch17_graphes as g

## 1. Le modèle de Ising

Soit $G = (V, E)$ un graphe sur $p$ sommets et $X = (X_1, \dots, X_p) \in \mathcal X = \{0,1\}^p$.
Le modèle de Ising définit la loi jointe (éq. (17.28)–(17.29) du livre)

$$
p(x, \Theta) = \exp\Big[\sum_{(j,k)\in E}\theta_{jk}x_jx_k - \Phi(\Theta)\Big],
\qquad
\Phi(\Theta) = \log\sum_{x\in\mathcal X}\exp\Big[\sum_{(j,k)\in E}\theta_{jk}x_jx_k\Big].
$$

$\Phi$ est le **logarithme de la fonction de partition** : il assure que les $2^p$
probabilités somment à 1. Seules des interactions **par paires** sont modélisées : c'est
un graphe de Markov par paires, avec le potentiel $\psi_{jk}(x_j, x_k) = e^{\theta_{jk}x_jx_k}$
sur chaque arête.

**Le nœud constant.** Pour des raisons techniques (exercice 17.10), on ajoute un nœud
$X_0 \equiv 1$ relié à tous les autres : le terme $\theta_{j0}x_jx_0 = \theta_{j0}x_j$ joue
le rôle d'un effet principal (une « ordonnée à l'origine » propre à chaque variable).
Sans lui, le modèle à $\Theta = 0$ serait forcément la loi uniforme, et les marges ne
pourraient pas être ajustées librement. Nous écrivons donc

$$
\log p(x) = \sum_{j=1}^p \theta_{j0}x_j + \sum_{j<k}\theta_{jk}x_jx_k - \Phi(\Theta),
$$

avec $\theta_{jk} = 0$ si $(j,k)\notin E$. Dans le code, `theta0` est le vecteur des
$\theta_{j0}$ et `Theta` la matrice symétrique des $\theta_{jk}$ (diagonale nulle).